# HADIS — YOLOv8 Drone Visual Detection Training

**High Altitude Drone Intelligence System**  
Author: Prakash Tiwari | Chandigarh Engineering College (IKGPTU)

This notebook trains a YOLOv8m model for multi-class drone detection using the
HADIS processed dataset on Google Drive. Supports resume-from-checkpoint.

---

In [ ]:
# Cell 2 — Install dependencies
!pip install -q ultralytics

In [ ]:
# Cell 3 — Mount Google Drive and clone HADIS repo
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/HADIS'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tiwari1782/HADIS.git {REPO_DIR}
    print(f'[HADIS] Repository cloned to {REPO_DIR}')
else:
    print(f'[HADIS] Repository already exists at {REPO_DIR}')

In [ ]:
# Cell 4 — Imports and configuration
import sys
import os
import shutil

sys.path.append('/content/HADIS')
from config import PATHS, HYPERPARAMS, DRONE_CLASSES, THREAT_LEVELS, NUM_CLASSES

import torch
from ultralytics import YOLO

print(f'[HADIS] Drone classes ({NUM_CLASSES}): {DRONE_CLASSES}')
print(f'[HADIS] Threat levels: {THREAT_LEVELS}')
print(f'[HADIS] YOLOv8 training config:')
print(f'        Epochs:    {HYPERPARAMS["yolo_epochs"]}')
print(f'        Img size:  {HYPERPARAMS["yolo_img_size"]}')
print(f'        Batch:     {HYPERPARAMS["yolo_batch"]}')
print(f'        Patience:  {HYPERPARAMS["yolo_patience"]}')

In [ ]:
# Cell 5 — Verify GPU availability
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'[HADIS] GPU available: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('[HADIS] WARNING: No GPU detected. Training will be extremely slow.')
    print('[HADIS] Go to Runtime > Change runtime type > GPU')

In [ ]:
# Cell 6 — Load model with resume-from-checkpoint logic
weights_dir = PATHS['weights_yolo']
os.makedirs(weights_dir, exist_ok=True)

checkpoint_path = os.path.join(weights_dir, 'last.pt')
resume_training = False

if os.path.exists(checkpoint_path):
    print(f'[HADIS] Checkpoint found at {checkpoint_path}')
    print('[HADIS] Resuming training from last checkpoint...')
    model = YOLO(checkpoint_path)
    resume_training = True
else:
    print('[HADIS] No checkpoint found. Starting fresh with yolov8m.pt')
    model = YOLO('yolov8m.pt')
    resume_training = False

print(f'[HADIS] Model loaded | Resume: {resume_training}')

In [ ]:
# Cell 7 — Train YOLOv8
dataset_yaml = os.path.join('/content/HADIS', 'hadis-ml', 'yolov8', 'dataset.yaml')

try:
    results = model.train(
        data=dataset_yaml,
        epochs=HYPERPARAMS['yolo_epochs'],
        imgsz=HYPERPARAMS['yolo_img_size'],
        batch=HYPERPARAMS['yolo_batch'],
        patience=HYPERPARAMS['yolo_patience'],
        project=weights_dir,
        name='hadis_yolov8_run',
        exist_ok=True,
        resume=resume_training,
        save=True,
        save_period=1,
        device=0 if torch.cuda.is_available() else 'cpu',
        verbose=True,
    )
    print('[HADIS] Training completed successfully.')
except Exception as e:
    print(f'[HADIS] Training interrupted: {e}')
    print('[HADIS] You can resume by re-running this notebook.')

In [ ]:
# Cell 8 — Copy best weights to Drive
run_dir = os.path.join(weights_dir, 'hadis_yolov8_run', 'weights')
src_best = os.path.join(run_dir, 'best.pt')
dst_best = os.path.join(weights_dir, 'hadis_yolov8_best.pt')

try:
    if os.path.exists(src_best):
        shutil.copy2(src_best, dst_best)
        print(f'[HADIS] Best weights saved to: {dst_best}')
    else:
        print(f'[HADIS] WARNING: best.pt not found at {src_best}')
        print('[HADIS] Checking for last.pt instead...')
        src_last = os.path.join(run_dir, 'last.pt')
        if os.path.exists(src_last):
            shutil.copy2(src_last, dst_best)
            print(f'[HADIS] Last checkpoint saved as best to: {dst_best}')
except Exception as e:
    print(f'[HADIS] Error copying weights: {e}')

In [ ]:
# Cell 9 — Validate model and print metrics
try:
    best_model = YOLO(dst_best)
    val_results = best_model.val(
        data=dataset_yaml,
        imgsz=HYPERPARAMS['yolo_img_size'],
        batch=HYPERPARAMS['yolo_batch'],
        device=0 if torch.cuda.is_available() else 'cpu',
    )
    
    map50 = val_results.box.map50
    map50_95 = val_results.box.map
    
    print(f'[HADIS] ========== Validation Results ==========')
    print(f'[HADIS] mAP@50:      {map50:.4f}')
    print(f'[HADIS] mAP@50-95:   {map50_95:.4f}')
    print(f'[HADIS] =========================================') 
    print(f'[HADIS] Best weights: {dst_best}')
except Exception as e:
    print(f'[HADIS] Validation error: {e}')